![Logotipo de LVMM](https://lvmm.mx/wp-content/uploads/2019/05/LVMM_logo_500-90x90.png)

# Guía sencilla para usar el clúster LVMM

¡Bienvenido/a! Esta guía explica cómo conectarse al clúster, mover archivos y ejecutar programas de cálculo. No hace falta conocer todos los detalles de SLURM para empezar: siga los pasos y adapte los ejemplos a su cuenta y a su trabajo.

> **Antes de empezar:** los nombres de servidores, particiones, programas y recursos pueden cambiar. Si un ejemplo no coincide con lo que ve al conectarse, consulte `sinfo`, `module avail` o a la administración del clúster.

## Ruta rápida
1. [Conéctese al clúster](#ssh).
2. [Suba sus archivos](#sftp).
3. [Prepare el programa](#module).
4. [Envíe el cálculo](#slurm); para la mayoría de los trabajos, use `sbatch`.
5. [Revise el trabajo](#squeue) y consulte sus archivos de salida.

<details>
<summary>Ver la opción técnica y detalles para usuarios con experiencia</summary>

### Referencia técnica rápida

- `ssh usuario@servidor`: abre una sesión en el nodo de acceso. No ejecute cálculos intensivos directamente allí.
- `sinfo`: consulta particiones y estados de nodos; `module avail`: consulta módulos cargables.
- `sbatch archivo.job`: envía un trabajo por lotes; `srun`: inicia tareas dentro de los recursos asignados.
- `squeue --user $USER`: muestra los trabajos de la cuenta actual; `scancel JOBID`: cancela un trabajo concreto.
- Ajuste partición, tareas, memoria, tiempo y aceleradores de acuerdo con la configuración del sitio y los requisitos de su aplicación. No asuma que las opciones de GPU son iguales en todos los clústeres.

Consulte las secciones siguientes y la documentación de SLURM para detalles.

</details>

---

<a name="ssh"></a>
## 1. Conectarse al clúster

### ¿Qué necesita?
Su cuenta autorizada y acceso a la red del CNyN. Si no tiene cuenta o no puede conectarse, solicite ayuda a la administración del clúster.

### Conexión
Desde una terminal de su computadora, ejecute (cambie `usuario` por su nombre de cuenta):

```bash
ssh usuario@192.168.100.237
```

Escriba su contraseña cuando se le solicite. Al entrar, verá un mensaje de bienvenida y un indicador parecido a `usuario@lvmm:~$`.

> **Importante:** este es el nodo de acceso. Úselo para preparar archivos y enviar trabajos, no para realizar cálculos intensivos. La dirección IP del ejemplo puede cambiar; confirme los datos vigentes con la administración.

<a name="sftp"></a>
## 2. Transferir archivos

Puede subir o descargar archivos desde la terminal de su computadora. Reemplace `usuario` y las rutas de ejemplo por los datos de su cuenta.

### Subir un archivo al clúster

```bash
scp archivo.txt usuario@192.168.100.237:/tmpu/grupo/usuario/
```

### Descargar una carpeta del clúster

La opción `-r` permite copiar carpetas y su contenido:

```bash
scp -r usuario@192.168.100.237:/tmpu/grupo/usuario/datos .
```

El punto final `.` significa “la carpeta actual” de su computadora. También puede abrir una sesión interactiva con `sftp usuario@192.168.100.237`; escriba `help` para ver sus comandos.

> **Consejo:** compruebe que la ruta de destino existe y que tiene permiso para escribir en ella.

<a name="module"></a>
## 3. Preparar el programa

Los programas instalados se habilitan mediante módulos. Primero, consulte qué hay disponible:

```bash
module avail
```

Busque el nombre y la versión del programa que necesita. Por ejemplo, para cargar la versión de VASP para GPU que aparece en el listado:

```bash
ml load vasp/6.4.3-nvhpc-gpu
```

Es normal que aparezcan mensajes indicando que también se cargaron dependencias.

> **Importante:** cargar un módulo no reserva recursos ni autoriza a ejecutar un cálculo grande en el nodo de acceso. Use SLURM para ejecutar programas en los nodos de cómputo. La solicitud de GPU depende de la configuración local; confirme cómo pedirla antes de enviar el trabajo.

<details>
<summary>Detalles técnicos: módulos y entorno</summary>

`module avail` (o su abreviatura `ml av`) lista los módulos visibles en el entorno. Cargue la variante compatible con el compilador, MPI y tipo de hardware que requiere su programa. Para trabajos por lotes, inicialice el entorno de módulos dentro del script si el sistema lo requiere; en los ejemplos de este clúster se usa `. /etc/profile.d/modules.sh`.

</details>

<a name="slurm"></a>
## 4. Enviar y gestionar cálculos con SLURM

SLURM organiza el uso de los nodos de cálculo. Para un trabajo que puede tardar, lo habitual es enviarlo a la cola con `sbatch`; así no necesita mantener la terminal abierta.

### 4.1 Ver qué recursos hay: `sinfo`

```bash
sinfo
```

El resultado muestra particiones y disponibilidad. Consulte la salida actual para elegir una partición adecuada; los ejemplos de nombres y estados pueden cambiar.

<details>
<summary>Detalles técnicos: interpretar `sinfo`</summary>

`idle` suele indicar recursos disponibles, `mix` indica que hay recursos disponibles y ocupados en el nodo, y `alloc` que los recursos están asignados. La interpretación exacta puede depender de la configuración de SLURM. Para un resumen por nodo, pruebe `sinfo --Node` o `sinfo -N`.

</details>

### 4.2 Ejecutar con `srun`

` srun ` inicia una aplicación usando recursos administrados por SLURM. Este ejemplo solicita 16 tareas en la partición `q_mem` (confirme que estos valores se ajustan a su cálculo):

```bash
srun --partition=q_mem --ntasks=16 vasp_ncl
```

El comando puede esperar hasta que haya recursos disponibles y la terminal permanecerá ocupada mientras se ejecuta. Para trabajos largos, suele ser más cómodo usar `sbatch`.

### 4.3 Enviar un trabajo con `sbatch`

Cree un archivo llamado `vasp.job` con un contenido como este. Los recursos y el módulo son un ejemplo: verifique que sean apropiados para su aplicación y para la configuración vigente.

```bash
#!/usr/bin/env bash
#SBATCH --job-name=VASP_test
#SBATCH --nodes=1
#SBATCH --ntasks=16
#SBATCH --cpus-per-task=1
#SBATCH --output=vasp_test-%j.out
#SBATCH --error=vasp_test-%j.err
#SBATCH --partition=q_cpu

. /etc/profile.d/modules.sh
ml vasp/6.4.3-intel
srun vasp_ncl
```

Envíelo desde la carpeta donde están el script y los archivos de entrada:

```bash
sbatch vasp.job
```

SLURM devolverá un número de trabajo (`JOBID`). Los archivos de salida y error se escribirán con ese número en el nombre (`%j`).

> **Antes de enviar:** confirme la partición, el módulo y el número de tareas. Para GPU, use únicamente las opciones de solicitud indicadas por la administración del clúster.

<details>
<summary>Detalles técnicos: solicitudes de recursos</summary>

Las directivas `#SBATCH` deben estar al inicio del script, antes de los comandos ejecutables. En este ejemplo, `--nodes` solicita nodos, `--ntasks` tareas MPI y `--cpus-per-task` CPU por tarea. Añada solicitudes de memoria, tiempo máximo o aceleradores solo con los valores y opciones admitidos por la configuración local. `srun` dentro del script inicia el paso de trabajo usando la asignación de `sbatch`. Evite solicitar más recursos de los que requiere el cálculo.

</details>

### 4.4 Revisar el estado: `squeue`

Para ver los trabajos de su cuenta, ejecute:

```bash
squeue --user $USER
```

`R` significa que el trabajo está en ejecución; `PD` significa que está pendiente. Si aparece pendiente, revise la razón que muestra SLURM y espere o consulte a la administración si no está claro.

### 4.5 Cancelar un trabajo: `scancel`

Si necesita detener un trabajo, use su `JOBID` real (no escriba literalmente `JOBID`):

```bash
scancel 12345
```

> Verifique cuidadosamente el identificador antes de cancelar.

### 4.6 Sesión interactiva: `salloc`

` salloc ` reserva recursos para pruebas interactivas. Por ejemplo:

```bash
salloc -p q_cpu -N 1 -n 16 -J prueba -t 60
```

Cuando se conceda la asignación, cargue el módulo necesario e inicie el programa con `srun`. Al terminar, escriba `exit` o presione `Ctrl + D` para cerrar la sesión. Para cálculos largos, prefiera `sbatch`.

<details>
<summary>Detalles técnicos: estados y flujo de ejecución</summary>

Estados de SLURM que puede encontrar: `PD` (pendiente), `R` (en ejecución), `CG` (finalizando) y `CD` (completado). `salloc` solicita una asignación interactiva; según la configuración, puede abrir o no automáticamente un shell en el nodo asignado. Ejecute las aplicaciones con `srun` para que SLURM gestione el paso de trabajo. Para diagnosticar un trabajo pendiente, inspeccione la razón en `squeue` y consulte las políticas y límites locales.

</details>

<a name="references"></a>
---
## 5. Referencias y ayuda

- [Sitio web de SLURM](https://slurm.schedmd.com/)
- [Documentación de SLURM](https://slurm.schedmd.com/documentation.html)
- [Comunidad de usuarios de SLURM](https://groups.google.com/g/slurm-users)
- [Environment Modules](https://modules.sourceforge.net/)

Si un comando no funciona o no conoce la partición adecuada, contacte a la administración del clúster e incluya el mensaje de error y el `JOBID`, si corresponde.

Aldo Rodríguez Guerrero, 2024–2026